# Data Visualizations

In [ ]:
import pandas as pd
import numpy as np

import plotly.graph_objects as go
import statsmodels.api as sm
import plotly.express as px

In [ ]:
df = pd.read_csv("./data/processed/diabetes_food_clean.csv")
df.head()

# Visualization 1 - Poverty Rate vs Diabetes Rate

**Poorer counties tend to have higher diabetes rates.**  

Each dot is a county, and each line shows the trend within one state. Click a state in the legend to hide it, or double-click to show only that state. Zoom in to explore the crowded area on the left.

In [ ]:
fig = px.scatter(
    df,
    x = "PovertyRatePerc",
    y = "DiabetesRate",
    color = "State",
    hover_data = df.columns,
    trendline = "ols",
    title = "Poverty Rate vs Diabetes Rate",
    labels = {
        'PovertyRatePerc': 'Poverty Rate',
        'DiabetesRate': 'Diabetes Rate'
    },
    height = 750
)

x_min = df["PovertyRatePerc"].min()
x_max = df["PovertyRatePerc"].max()

y_min = df["DiabetesRate"].min()
y_max = df["DiabetesRate"].max()

fig.update_xaxes(range = [x_min - 0.01, x_max + 0.01])
fig.update_yaxes(range = [y_min - 0.01, y_max + 0.01])

fig.show()

# Visualization 2 - Diabetes Rate Per State
**Where you live matters: diabetes rates vary a lot across the country.**  

Darker colors mean higher average diabetes rates. Hover over a state to see its exact rate.

In [ ]:
state_abbr = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "District of Columbia": "DC", "Florida": "FL", "Georgia": "GA", "Hawaii": "HI",
    "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA",
    "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME",
    "Maryland": "MD", "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN",
    "Mississippi": "MS", "Missouri": "MO", "Montana": "MT", "Nebraska": "NE",
    "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM",
    "New York": "NY", "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH",
    "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI",
    "South Carolina": "SC", "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX",
    "Utah": "UT", "Vermont": "VT", "Virginia": "VA", "Washington": "WA",
    "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY"
}

In [ ]:
# Population-weighted diabetes prevalence by state
df_state = (
    df.assign(
        WeightedDiabetes=df['DiabetesRate'] * df['Pop2010']
    )
    .groupby('State')
    .agg(
        WeightedDiabetes=('WeightedDiabetes', 'sum'),
        TotalPopulation=('Pop2010', 'sum')
    )
    .reset_index()
)

df_state['DiabetesRate'] = (
    df_state['WeightedDiabetes'] / df_state['TotalPopulation']
)

df_state['StateAbbr'] = df_state['State'].map(state_abbr)

In [ ]:

fig = px.choropleth(
    df_state,
    locations='StateAbbr',
    locationmode='USA-states',
    color='DiabetesRate',
    hover_name='State',
    scope='usa',
    color_continuous_scale='Viridis',
    title='Average US Diabetes Rate by State',
    labels={'DiabetesRate': 'Diabetes Rate'},
)

fig.update_layout(
    width=1000,
    height=650,
    margin=dict(l=20, r=20, t=70, b=20),
    coloraxis_colorbar=dict(tickformat=".0%")
)

fig.show()

**Takeaway:** Diabetes rates are highest across the South and Appalachia, with Mississippi, Alabama, and West Virginia at the top (roughly 14%). Colorado, Wyoming, and parts of New England sit at the bottom, around 9%. That gap of about five to six percentage points between the highest and lowest states shows how much geography matters.

*Note: each state's diabetes rate is a population-weighted average of its counties using county population. Counties with larger populations therefore contribute more to the state estimate.*

# Visualization 3 - Features vs Diabetes Rate

**Which community traits go with higher diabetes rates, and does food access?**

Use the dropdown to switch between ten county characteristics. Each dot is a county, and the line shows the overall trend. Hover over a dot to see its state. Zoom in to look at clusters, and double-click to reset.

In [ ]:
features = [
    "FoodAccessScore",
    "Urban",
    "SeniorsRate",
    "GroupQuarters",
    "PovertyRatePerc",
    "BlackRate",
    "AsianRate",
    "NHOPIRate",
    "AIANRate",
    "HispanicRate"
]

labels = {
    "FoodAccessScore": "Food Access Score (0-4)",
    "Urban": "Share of Population in Urban Areas",
    "SeniorsRate": "Share of Population Age 65+",
    "GroupQuarters": "Share Living in Group Quarters",
    "PovertyRatePerc": "Poverty Rate",
    "BlackRate": "Share Black",
    "AsianRate": "Share Asian",
    "NHOPIRate": "Share Native Hawaiian / Pacific Islander",
    "AIANRate": "Share American Indian / Alaska Native",
    "HispanicRate": "Share Hispanic"
}


In [ ]:

fig = go.Figure()
for i, feature in enumerate(features):

    # Remove missing values
    temp = df[[feature, "DiabetesRate"]].dropna()

    x = temp[feature]
    y = temp["DiabetesRate"]

    # OLS regression
    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()

    # Points
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode="markers",
            name=feature,
            visible=(i == 0),
            text=df.loc[temp.index, "State"],
            hovertemplate=
                f"{feature}: %{{x}}<br>" +
                "Diabetes Rate: %{y}<br>" +
                "State: %{text}<extra></extra>"
        )
    )

    # Regression line
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = model.params.iloc[0] + model.params.iloc[1] * x_line

    fig.add_trace(
        go.Scatter(
            x=x_line,
            y=y_line,
            mode="lines",
            name="OLS Trendline",
            visible=(i == 0),
            hoverinfo="skip"
        )
    )


In [ ]:

# Dropdown
buttons = []

for i, feature in enumerate(features):

    visible = [False] * (len(features) * 2)
    visible[i * 2] = True
    visible[i * 2 + 1] = True

    buttons.append(
        dict(
            label=labels[feature],
            method="update",
            args=[
                {"visible": visible},
                {
                    "title": f"{labels[feature]} vs Diabetes Rate",
                    "xaxis.title.text": labels[feature],
                    "xaxis.autorange": True,   # resets zoom when you switch features
                    "xaxis.tickformat": "" if feature == "FoodAccessScore" else ".0%"
                }
            ]
        )
    )


In [ ]:

fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True
        )
    ],
    xaxis_title=labels[features[0]],
    yaxis_title="Diabetes Rate",
    title=f"{features[0]} vs Diabetes Rate",
    height = 750
)

fig.show()

**Takeaway:** Older age, higher poverty, and some demographic groups go with higher diabetes rates. Food access is the exception: it has by far the smallest effect, about 0.16 points across its whole 0-4 scale, compared with roughly 3 points for a 10-point higher share of seniors.

*Note: this shows association, not cause.*

# Visualization 4 - Digging Into the States

**A state's average hides a lot of variation between its counties.**

Each box is sized by population and colored by diabetes rate. Click a state to zoom into its counties, and click the bar at the top to zoom back out. Hover over any box for its exact rate.

In [ ]:
fig = px.treemap(
    df,
    path=[px.Constant("United States"), "State", "County"],
    values="Pop2010",
    color="DiabetesRate",
    color_continuous_scale="Viridis",
    range_color=[0.08, 0.16],   # only state-level colors stay readable
    maxdepth=2,                 # show only states at first
    title="Click a state to explore its counties",
)

fig.update_traces(
    hovertemplate="<b>%{label}</b><br>"
                  "Diabetes rate: %{color:.1%}<br>"
                  "Population: %{value:,}<extra></extra>",
    textinfo="label",
    root_color="lightgrey",
)

fig.update_coloraxes(colorbar_title="Diabetes Rate", colorbar_tickformat=".0%")
fig.update_layout(height=650, margin=dict(l=10, r=10, t=60, b=10))

fig.show()

**Takeaway:** County diabetes rates run from about 5% to nearly 25%, and that spread exists inside states too. In a typical state, counties span about 8 points, and Texas alone runs from 8% (Travis County) to nearly 25% (Presidio County). Our models account for both layers. The state matters: Colorado's counties average about 6 points below Alabama's, and still about 3.7 points below after accounting for age, poverty, demographics, urbanicity, and food access. Local traits matter too: within the same state, counties with more poverty and more seniors tend to have higher diabetes rates.

*Note: box size is total population. Each state's color is a population-weighted average of its counties.*

## Parallel Coordinates: County Profiles

**Counties with the lowest diabetes rates share a profile: fewer seniors, lower poverty, and more urban residents.**

Each line is one county. The chart opens with the 10% of counties with the lowest diabetes rates selected, so you can see where they fall on every other trait. Drag on any axis to select a different range, and the lines passing through it stay highlighted. You can select on several axes at once, and drag an axis header to reorder them. Click an axis once to clear its selection.

In [ ]:
df_sorted = df.sort_values("DiabetesRate")

cols = {
    "DiabetesRate": "Diabetes Rate",
    "SeniorsRate": "Seniors Share",
    "PovertyRatePerc": "Poverty Rate",
    "BlackRate": "Black Share",
    "HispanicRate": "Hispanic Share",
    "Urban": "Urban Share",
    "FoodAccessScore": "Food Access Score",
}

dims = [
    dict(
        label=label,
        values=df[col],
        tickformat=".1f" if col == "FoodAccessScore" else ".0%",
    )
    for col, label in cols.items()
]

# Pre-select the lowest 5% of counties by diabetes rate
cutoff = df["DiabetesRate"].quantile(0.10)
dims[0]["constraintrange"] = [df["DiabetesRate"].min(), cutoff]


In [ ]:

fig = go.Figure(
    go.Parcoords(
        line=dict(
            color=df["DiabetesRate"],
            colorscale="Viridis",
            showscale=True,
            colorbar=dict(title="Diabetes Rate", tickformat=".0%"),
        ),
        dimensions=dims,
    )
)

fig.update_layout(
    title="What do the counties with the lowest diabetes rates have in common?",
    height=600,
    margin=dict(l=80, r=40, t=80, b=40),
)
fig.show()

**Takeaway:** Select the counties with the lowest diabetes rates and a pattern shows up on the other axes. Compared with the highest-rate counties, they have much lower poverty (about 10% vs. 25% at the median) and are far more urban (about 72% of the population in urban tracts vs. 16%). They also have fewer seniors than a typical county, while the highest-rate counties have a much larger Black population share (about 34% vs. 2%). This matches our regression, where poverty, seniors share, and race and ethnicity shares (relative to a county with more White residents) had positive effects and urban share a negative one. Food access barely differs between the groups, which fits its near-zero effect in the model.

*Note: the chart opens with the lowest 10% of counties by diabetes rate selected. Each axis is scaled to its own minimum and maximum, so heights aren't comparable across axes. The regression holds the other traits and state fixed, while this chart looks at all of them at once, so the two can differ. Race and ethnicity shares are associations that likely reflect factors not in the model, such as income, healthcare access, and local history, and they don't mean that race causes diabetes.*

# Visualization Exports

In [ ]:
# Viz 1: Poverty Rate vs Diabetes Rate (from cell 4)
fig.write_html("visuals/poverty_vs_diabetes.html", include_plotlyjs="cdn")

# Viz 2: Diabetes Rate Per State (from cell 8)
fig.write_html("visuals/diabetes_by_state.html", include_plotlyjs="cdn")

# Viz 3: Features vs Diabetes Rate (from in cell 14)
fig.write_html("visuals/features_vs_diabetes.html", include_plotlyjs="cdn")

# Viz 4: Digging Into the States / treemap (from in cell 17)
fig.write_html("visuals/state_treemap.html", include_plotlyjs="cdn")

# Viz 5: Parallel Coordinates (from in cell 21)
fig.write_html("visuals/parallel_coords.html", include_plotlyjs="cdn")